## 01 — Análise Exploratória dos Dados

**Projeto:** Propensão de compra em e-commerce  
**Pós-Tech:** Machine Learning Engineering — Fase 2  
**Grupo:** 56

### Objetivo

Explorar os dados de navegação em e-commerce para compreender
o comportamento dos visitantes e identificar características
associadas à realização de uma compra.

A análise contempla:

- Estrutura e tipos dos dados.
- Valores ausentes e registros duplicados.
- Distribuição da variável-alvo.
- Distribuição das variáveis numéricas e categóricas.
- Relações entre as características dos visitantes e a compra.
- Cuidados e hipóteses para as próximas etapas de modelagem.

### Cuidados metodológicos

Este notebook não modifica o dataset original.

Análises que orientem a seleção de variáveis ou decisões de
modelagem devem utilizar o conjunto de treinamento, preservando
os conjuntos de validação e teste.

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO AMBIENTE E IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from IPython.display import display


# ============================================================
# RÓTULOS DAS VARIÁVEIS EM PORTUGUÊS
# ============================================================

VARIABLE_LABELS = {
    "Administrative": "Quantidade de páginas administrativas",
    "Administrative_Duration": "Tempo em páginas administrativas",
    "Informational": "Quantidade de páginas informativas",
    "Informational_Duration": "Tempo em páginas informativas",
    "ProductRelated": "Quantidade de páginas de produtos",
    "ProductRelated_Duration": "Tempo em páginas de produtos",
    "BounceRates": "Taxa de rejeição",
    "ExitRates": "Taxa de saída",
    "PageValues": "Valor das páginas",
    "SpecialDay": "Proximidade de data especial",
    "Month": "Mês",
    "OperatingSystems": "Sistema operacional",
    "Browser": "Navegador",
    "Region": "Região",
    "TrafficType": "Tipo de tráfego",
    "VisitorType": "Tipo de visitante",
    "Weekend": "Fim de semana",
    "Revenue": "Compra realizada",
}
# ============================================================
# CONFIGURAÇÕES DE VISUALIZAÇÃO
# ============================================================

plt.style.use("ggplot")

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "figure.dpi": 100,
    }
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


# ============================================================
# LOCALIZAÇÃO DA RAIZ DO PROJETO
# ============================================================

CURRENT_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (
            CURRENT_DIRECTORY,
            *CURRENT_DIRECTORY.parents,
        )
        if (directory / "pyproject.toml").is_file()
        and (directory / "data").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto. "
        "Abra o notebook dentro de ecommerce-purchase-propensity-mlops."
    )


# ============================================================
# LOCALIZAÇÃO DO DATASET
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT / "data" / "raw" / "online_shoppers_intention.csv"
)

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset não encontrado: {DATASET_PATH}. "
        "Verifique se o arquivo está disponível em data/raw."
    )

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_PATH}")

---

### 1. Carregamento e inspeção inicial

Nesta etapa, carregamos o dataset original e verificamos suas
dimensões, os nomes das colunas e uma amostra dos registros.

Essa inspeção permite conferir se o arquivo foi lido corretamente
antes de iniciar as análises.

In [ ]:
# ============================================================
# CARREGAMENTO DO DATASET ORIGINAL
# ============================================================

df_raw = pd.read_csv(DATASET_PATH)


# ============================================================
# DIMENSÕES DO DATASET
# ============================================================

print(f"Quantidade de registros: {df_raw.shape[0]:,}")
print(f"Quantidade de colunas: {df_raw.shape[1]}")


# ============================================================
# VISUALIZAÇÃO DOS PRIMEIROS REGISTROS
# ============================================================

display(df_raw.head())


# ============================================================
# NOMES DAS COLUNAS
# ============================================================

display(
    pd.DataFrame(
        {
            "Coluna": df_raw.columns,
            "Tipo": df_raw.dtypes.astype(str).to_numpy(),
        }
    )
)

---

### 2. Verificação da qualidade dos dados

Nesta etapa, verificamos:

- Valores ausentes por coluna.
- Quantidade de valores distintos.
- Registros completamente duplicados.
- Colunas sem variação.

Os resultados serão utilizados para documentar possíveis problemas
de qualidade. Nenhum registro será removido automaticamente:
linhas iguais podem representar sessões distintas.

In [ ]:
# ============================================================
# RESUMO DA QUALIDADE DOS DADOS
# ============================================================

quality_summary = pd.DataFrame(
    {
        "Tipo": df_raw.dtypes.astype(str),
        "Valores Ausentes": df_raw.isna().sum(),
        "Percentual Ausentes": df_raw.isna().mean().mul(100),
        "Valores Distintos": df_raw.nunique(dropna=True),
    }
)

quality_summary.index.name = "Coluna"

display(quality_summary)


# ============================================================
# VERIFICAÇÃO DE REGISTROS DUPLICADOS
# ============================================================

# Conta as repetições além da primeira ocorrência de cada linha.
duplicate_count = int(df_raw.duplicated().sum())
duplicate_percentage = duplicate_count / len(df_raw) * 100

print(f"Registros duplicados além da primeira ocorrência: {duplicate_count:,}")
print(f"Percentual de registros duplicados: {duplicate_percentage:.2f}%")


# ============================================================
# VERIFICAÇÃO DE COLUNAS SEM VARIAÇÃO
# ============================================================

constant_columns = df_raw.columns[
    df_raw.nunique(dropna=False) <= 1
].tolist()

print(f"Colunas sem variação: {constant_columns or 'Nenhuma'}")


# ============================================================
# AMOSTRA DOS GRUPOS DE REGISTROS DUPLICADOS
# ============================================================

if duplicate_count > 0:
    # Inclui todas as ocorrências dos grupos, inclusive a primeira.
    display(
        df_raw.loc[df_raw.duplicated(keep=False)].head(10)
    )

#### Resultados da verificação de qualidade

O dataset contém 12.330 registros e 18 colunas, sem valores ausentes identificados pelo pandas e sem colunas constantes.

Foram encontradas 125 repetições completas além da primeira ocorrência de cada linha, correspondentes a aproximadamente 1,01% dos registros.

Esses registros serão preservados nesta análise, pois a igualdade entre os valores não comprova que representam a mesma sessão.

Antes da modelagem, será necessário verificar o tratamento dessas repetições e sua possível presença em diferentes conjuntos de dados, documentando a decisão adotada.

---

### 3. Distribuição da variável-alvo

A variável Revenue indica se a sessão resultou em compra:

- False: sessão sem compra.
- True: sessão com compra.

Vamos calcular a quantidade e a proporção de cada classe para avaliar o equilíbrio entre sessões com e sem compra.

Esta distribuição descreve a base original. Decisões de modelagem deverão ser apoiadas nas análises do conjunto de treinament

In [ ]:
# ============================================================
# DISTRIBUIÇÃO DA VARIÁVEL-ALVO
# ============================================================

TARGET_COLUMN = "Revenue"

target_counts = (
    df_raw[TARGET_COLUMN]
    .value_counts()
    .reindex([False, True], fill_value=0)
)

target_summary = pd.DataFrame(
    {
        "classe": ["Sem compra", "Com compra"],
        "quantidade": target_counts.to_numpy(),
        "percentual": target_counts.to_numpy() / len(df_raw) * 100,
    }
)

display(target_summary)


# ============================================================
# VISUALIZAÇÃO DA DISTRIBUIÇÃO DAS CLASSES
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    target_summary["classe"],
    target_summary["quantidade"],
    color=["#64748B", "#2563EB"],
)

labels = [
    (
        f"{quantity:,}".replace(",", ".")
        + f" ({percentage:.2f}%)".replace(".", ",")
    )
    for quantity, percentage in zip(
        target_summary["quantidade"],
        target_summary["percentual"],
    )
]

ax.bar_label(bars, labels=labels, padding=5)

ax.set_title("Distribuição das sessões com e sem compra")
ax.set_xlabel("Resultado da sessão")
ax.set_ylabel("Quantidade de sessões")
ax.set_ylim(0, target_summary["quantidade"].max() * 1.15)

fig.tight_layout()
plt.show()

#### Resultados da análise da variável-alvo

Das 12.330 sessões analisadas:

- 10.422 não resultaram em compra, correspondendo a 84,53%.
- 1.908 resultaram em compra, correspondendo a 15,47%.

A classe positiva, representada pelas sessões com compra, é minoritária. Existem aproximadamente 5,46 sessões sem compra para cada sessão com compra.

Um classificador que previsse exclusivamente a classe "sem compra" teria acurácia de 84,53% nesta base, mas recall zero para a classe positiva. Portanto, a acurácia isolada não é suficiente para avaliar o desempenho do modelo.

Na modelagem, será necessário avaliar métricas como precisão, recall, F1 e Average Precision, além da matriz de confusão.

O desbalanceamento não implica a necessidade automática de reamostragem. Eventuais estratégias serão comparadas utilizando os conjuntos de treinamento e validação.

---

### 4. Base de treinamento para as análises exploratórias

A inspeção anterior descreveu a estrutura, a qualidade e a distribuição da variável-alvo na base original.

A partir desta etapa, as análises de distribuições e relações entre variáveis utilizarão o conjunto de treinamento produzido pelo pipeline do projeto.

In [ ]:
# ============================================================
# LOCALIZAÇÃO DO CONJUNTO DE TREINAMENTO
# ============================================================

TRAIN_DATASET_PATH = (
    PROJECT_ROOT / "data" / "processed" / "train.parquet"
)

if not TRAIN_DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Conjunto de treinamento não encontrado: {TRAIN_DATASET_PATH}. "
        "Confira o nome do arquivo gerado pelo pipeline de pré-processamento."
    )


# ============================================================
# CARREGAMENTO DO CONJUNTO DE TREINAMENTO
# ============================================================

df_train = pd.read_parquet(TRAIN_DATASET_PATH)

if TARGET_COLUMN not in df_train.columns:
    raise ValueError(
        f"A variável-alvo '{TARGET_COLUMN}' não está no arquivo. "
        "Confira como o pipeline armazena as características e o alvo."
    )


# ============================================================
# RESUMO DA BASE DE TREINAMENTO
# ============================================================

print(
    "Quantidade de sessões: "
    f"{df_train.shape[0]:,}".replace(",", ".")
)

print(f"Quantidade de colunas: {df_train.shape[1]}")

print(
    f"Total de valores ausentes: "
    f"{df_train.isna().sum().sum()}"
)


# ============================================================
# AMOSTRA DOS REGISTROS EM PORTUGUÊS
# ============================================================

training_sample = df_train.head(5).copy()

training_sample[TARGET_COLUMN] = training_sample[TARGET_COLUMN].map(
    {
        False: "Sem compra",
        True: "Com compra",
    }
)

training_sample["Weekend"] = training_sample["Weekend"].map(
    {
        False: "Não",
        True: "Sim",
    }
)

training_sample["VisitorType"] = training_sample["VisitorType"].replace(
    {
        "Returning_Visitor": "Visitante recorrente",
        "New_Visitor": "Novo visitante",
        "Other": "Outro",
    }
)

training_sample["Month"] = training_sample["Month"].replace(
    {
        "Jan": "Janeiro",
        "Feb": "Fevereiro",
        "Mar": "Março",
        "Apr": "Abril",
        "May": "Maio",
        "June": "Junho",
        "Jul": "Julho",
        "Aug": "Agosto",
        "Sep": "Setembro",
        "Oct": "Outubro",
        "Nov": "Novembro",
        "Dec": "Dezembro",
    }
)

display(
    training_sample.rename(columns=VARIABLE_LABELS)
)


# ============================================================
# QUALIDADE E TIPOS DOS DADOS
# ============================================================

TYPE_LABELS = {
    "int64": "Inteiro",
    "float64": "Decimal",
    "bool": "Booleano",
    "object": "Texto / objeto",
    "str": "Texto",
    "string": "Texto",
}

training_quality = pd.DataFrame(
    {
        "Tipo de dado": (
            df_train.dtypes.astype(str).replace(TYPE_LABELS)
        ),
        "Valores ausentes": df_train.isna().sum(),
        "Ausentes (%)": df_train.isna().mean().mul(100),
        "Valores distintos": df_train.nunique(),
    }
)

display(
    training_quality
    .rename(index=VARIABLE_LABELS)
    .rename_axis("Variável")
)

---

### 5. Estatísticas descritivas das variáveis numéricas

A análise utiliza exclusivamente o conjunto de treinamento.

Serão examinadas contagens de páginas, durações, taxas, PageValues e SpecialDay.

As variáveis OperatingSystems, Browser, Region e TrafficType representam categorias codificadas como números e serão analisadas separadamente.

Calcularemos medidas de tendência central, dispersão e posição, além da proporção de valores iguais a zero. Essa análise ajuda a identificar concentração de valores, assimetria e diferenças de escala entre as variáveis.

Valores extremos não serão removidos automaticamente.

In [ ]:
# ============================================================
# DEFINIÇÃO DAS VARIÁVEIS NUMÉRICAS
# ============================================================

NUMERIC_COLUMNS = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
]


# ============================================================
# DEFINIÇÃO DAS VARIÁVEIS CATEGÓRICAS
# ============================================================

CATEGORICAL_COLUMNS = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]


# ============================================================
# ESTATÍSTICAS DESCRITIVAS DO CONJUNTO DE TREINAMENTO
# ============================================================

numeric_summary = (
    df_train[NUMERIC_COLUMNS]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])
    .T
    .rename(
        columns={
            "count": "Registros",
            "mean": "Média",
            "std": "Desvio padrão",
            "min": "Mínimo",
            "25%": "P25",
            "50%": "Mediana",
            "75%": "P75",
            "95%": "P95",
            "99%": "P99",
            "max": "Máximo",
        }
    )
)


# ============================================================
# PROPORÇÃO DE ZEROS E ASSIMETRIA
# ============================================================

numeric_summary["Percentual de zeros (%)"] = (
    df_train[NUMERIC_COLUMNS].eq(0).mean().mul(100)
)

numeric_summary["Assimetria"] = (
    df_train[NUMERIC_COLUMNS].skew()
)


# ============================================================
# REPRESENTAÇÃO DA TABELA
# ============================================================

numeric_summary_display = (
    numeric_summary
    .rename(index=VARIABLE_LABELS)
    .rename_axis("Variável")
)

display(numeric_summary_display)

#### Resultados das estatísticas descritivas

Todas as variáveis numéricas analisadas apresentam assimetria positiva, com diferentes intensidades.

Os tempos de navegação apresentam médias superiores às medianas e valores máximos elevados. O tempo em páginas de produtos possui a maior assimetria observada, de aproximadamente 8,51.

Há concentração expressiva em zero, especialmente nas variáveis:

- Proximidade de data especial: 89,63%.
- Tempo em páginas informativas: 80,52%.
- Quantidade de páginas informativas: 78,67%.
- Valor das páginas: 77,86%.

As variáveis apresentam escalas bastante diferentes. Essas características serão consideradas na avaliação de transformações e de pré-processamento, conforme o modelo utilizado.

Nenhum valor extremo será removido apenas com base nessas estatísticas. Os valores iguais a zero também serão preservados, pois não representam necessariamente dados ausentes.

---

### 6. Distribuição das variáveis numéricas

Os histogramas mostram a frequência dos valores no conjunto de treinamento, permitindo visualizar concentrações, dispersão e caudas das distribuições.

Nesta primeira visualização, será utilizada a escala original, sem cortes, exclusões ou transformações dos valores.

Cada gráfico possui sua própria escala.

In [ ]:
# ============================================================
# HISTOGRAMAS DAS VARIÁVEIS NUMÉRICAS
# ============================================================

fig, axes = plt.subplots(
    nrows=5,
    ncols=2,
    figsize=(16, 22),
)

for ax, column in zip(axes.flat, NUMERIC_COLUMNS):
    ax.hist(
        df_train[column],
        bins=30,
        color="#2563EB",
        edgecolor="white",
        linewidth=0.5,
    )

    ax.set_title(
        VARIABLE_LABELS.get(column, column),
        fontsize=13,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel("Valor observado", fontsize=11)
    ax.set_ylabel("Quantidade de sessões", fontsize=11)

    ax.tick_params(axis="both", labelsize=10)

    # Evita notação científica e deslocamentos nos eixos.
    ax.ticklabel_format(
        axis="both",
        style="plain",
        useOffset=False,
    )

    # Destaca a frequência com linhas de referência horizontais.
    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", visible=True, alpha=0.4)


# ============================================================
# REPRESETAÇÃO DA DISTRIBUIÇÃO DE VARIAVEIS
# ============================================================

fig.suptitle(
    "Distribuição das Variáveis Numéricas — Treinamento",
    fontsize=18,
    fontweight="bold",
)

fig.tight_layout(
    rect=[0, 0, 1, 0.97],
    h_pad=3,
    w_pad=2,
)

plt.show()

#### Resultados dos histogramas na escala original

As quantidades de páginas visitadas, os tempos de navegação e o valor das páginas apresentam concentração em valores baixos e caudas à direita.

Os valores elevados ampliam o eixo horizontal, dificultando a leitura da região onde se concentra a maior parte das sessões, principalmente nos tempos de navegação.

A taxa de rejeição e a taxa de saída apresentam concentrações próximas de 0,20. A proximidade de data especial apresenta valores discretos, com predominância do zero.

A primeira barra de cada histograma representa um intervalo. Sua altura não deve ser interpretada como a quantidade exata de valores iguais a zero.


#### Visualização complementar com transformação logarítmica

Para facilitar a leitura dos tempos de navegação e do valor das páginas, utilizaremos a transformação log1p(x), equivalente a ln(1 + x).

Essa transformação preserva os zeros e comprime a distância entre valores elevados. Todos os registros serão mantidos.

A transformação será utilizada exclusivamente na visualização. Sua adoção na modelagem dependerá de avaliação posterior.

---

In [ ]:
# ============================================================
# IMPORTAÇÃO PARA A VISUALIZAÇÃO LOGARÍTMICA
# ============================================================

import numpy as np


# ============================================================
# VARIÁVEIS PARA A VISUALIZAÇÃO COMPLEMENTAR
# ============================================================

LOG_VISUALIZATION_COLUMNS = [
    "Administrative_Duration",
    "Informational_Duration",
    "ProductRelated_Duration",
    "PageValues",
]


# ============================================================
# HISTOGRAMAS COM TRANSFORMAÇÃO LOG1P
# ============================================================

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(16, 10),
)

for ax, column in zip(axes.flat, LOG_VISUALIZATION_COLUMNS):
    # Transforma apenas os valores enviados ao gráfico.
    transformed_values = np.log1p(df_train[column])

    ax.hist(
        transformed_values,
        bins=30,
        color="#2563EB",
        edgecolor="white",
        linewidth=0.5,
    )

    ax.set_title(
        VARIABLE_LABELS.get(column, column),
        fontsize=13,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Valor transformado: ln(1 + x)",
        fontsize=11,
    )

    ax.set_ylabel(
        "Quantidade de sessões",
        fontsize=11,
    )

    ax.tick_params(axis="both", labelsize=10)

    # Evita notação científica e deslocamentos nos eixos.
    ax.ticklabel_format(
        axis="both",
        style="plain",
        useOffset=False,
    )

    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", visible=True, alpha=0.4)


# ============================================================
# REPRESETAÇÃO DA DISTRIBUIÇÃO DE VARIÁVEIS
# ============================================================

fig.suptitle(
    "Distribuição das Variáveis com Transformação Logarítmica — Treinamento",
    fontsize=17,
    fontweight="bold",
)

fig.tight_layout(
    rect=[0, 0, 1, 0.95],
    h_pad=3,
    w_pad=2,
)

plt.show()

#### Resultados da visualização com transformação logarítmica

A transformação log1p, equivalente a ln(1 + x), comprimiu os valores elevados e facilitou a visualização da distribuição dos valores positivos.

O efeito é especialmente visível no tempo em páginas de produtos, permitindo observar detalhes que estavam comprimidos na escala original.

O tempo em páginas administrativas, o tempo em páginas informativas e o valor das páginas continuam apresentando concentrações expressivas próximas de zero, coerentes com os percentuais de zeros calculados anteriormente.

A transformação preserva os zeros e não elimina essa concentração. As barras dos histogramas representam intervalos, e não contagens exclusivas de valores iguais a zero.

Os gráficos não comprovam normalidade nem determinam a adoção da transformação na modelagem. A transformação foi aplicada apenas à visualização, preservando os dados originais.

---

### 7. Distribuição das variáveis categóricas

Nesta etapa, calculamos a quantidade e o percentual de sessões por categoria no conjunto de treinamento.

OperatingSystems, Browser, Region e TrafficType são tratados como categorias, mesmo estando armazenados como números. Seus códigos não serão interpretados como medidas ou posições
em uma escala.

As tabelas permitem identificar categorias predominantes e categorias com poucos registros.

Esta etapa descreve a frequência das categorias. A relação com a realização de compra será analisada posteriormente.

In [ ]:
# ============================================================
# RÓTULOS DAS VARIÁVEIS CATEGÓRICAS 
# ============================================================

VARIABLE_LABELS.update(
    {
        "Month": "Mês",
        "OperatingSystems": "Sistema operacional",
        "Browser": "Navegador",
        "Region": "Região",
        "TrafficType": "Tipo de tráfego",
        "VisitorType": "Tipo de visitante",
        "Weekend": "Fim de semana",
    }
)


# ============================================================
# TRADUÇÃO DOS VALORES DAS CATEGORIAS
# ============================================================

CATEGORY_LABELS = {
    "Month": {
        "Jan": "Janeiro",
        "Feb": "Fevereiro",
        "Mar": "Março",
        "Apr": "Abril",
        "May": "Maio",
        "June": "Junho",
        "Jul": "Julho",
        "Aug": "Agosto",
        "Sep": "Setembro",
        "Oct": "Outubro",
        "Nov": "Novembro",
        "Dec": "Dezembro",
    },
    "VisitorType": {
        "Returning_Visitor": "Visitante recorrente",
        "New_Visitor": "Novo visitante",
        "Other": "Outro",
    },
    "Weekend": {
        False: "Não",
        True: "Sim",
    },
}


# ============================================================
# DISTRIBUIÇÃO DAS VARIÁVEIS CATEGÓRICAS
# ============================================================

category_summaries = {}
category_display_summaries = {}

for column in CATEGORICAL_COLUMNS:
    category_counts = df_train[column].value_counts(dropna=False)

    category_summary = (
        category_counts
        .rename("quantidade")
        .rename_axis("categoria")
        .reset_index()
    )

    category_summary["percentual"] = (
        category_summary["quantidade"] / len(df_train) * 100
    )

    # Preserva os valores originais para uso nas análises.
    category_summaries[column] = category_summary

    # Traduz somente a cópia utilizada na apresentação.
    category_display = category_summary.copy()

    if column in CATEGORY_LABELS:
        category_display["categoria"] = (
            category_display["categoria"]
            .replace(CATEGORY_LABELS[column])
        )

    category_display = category_display.rename(
        columns={
            "categoria": "Categoria",
            "quantidade": "Quantidade de sessões",
            "percentual": "Percentual (%)",
        }
    )

    category_display_summaries[column] = category_display

    print(f"\nVariável: {VARIABLE_LABELS.get(column, column)}")
    display(category_display)


# ============================================================
# GRÁFICOS DAS VARIÁVEIS CATEGÓRICAS
# ============================================================

for column in CATEGORICAL_COLUMNS:
    category_display = category_display_summaries[column]

    # Ajusta a altura conforme a quantidade de categorias.
    figure_height = max(3.5, len(category_display) * 0.42 + 1.5)

    fig, ax = plt.subplots(
        figsize=(12, figure_height),
    )

    positions = list(range(len(category_display)))

    bars = ax.barh(
        positions,
        category_display["Percentual (%)"],
        color="#2563EB",
        height=0.65,
    )

    category_labels = category_display["Categoria"].astype(str)

    # Explicita que os números representam códigos de categorias.
    if column in [
        "OperatingSystems",
        "Browser",
        "Region",
        "TrafficType",
    ]:
        category_labels = "Código " + category_labels

    ax.set_yticks(
        positions,
        labels=category_labels,
    )

    ax.invert_yaxis()


    # ========================================================
    # RÓTULOS COM PERCENTUAL E QUANTIDADE DE SESSÕES
    # ========================================================

    bar_labels = [
        (
            f"{percentage:.2f}%".replace(".", ",")
            + " — "
            + f"{quantity:,}".replace(",", ".")
            + " sessões"
        )
        for percentage, quantity in zip(
            category_display["Percentual (%)"],
            category_display["Quantidade de sessões"],
        )
    ]

    ax.bar_label(
        bars,
        labels=bar_labels,
        padding=6,
        fontsize=10,
    )


    # ========================================================
    # TÍTULOS E EIXOS
    # ========================================================

    ax.set_title(
        (
            "Distribuição por "
            + VARIABLE_LABELS.get(column, column).lower()
            + " — Treinamento"
        ),
        fontsize=14,
        fontweight="bold",
        pad=16,
    )

    ax.set_xlabel("Percentual de sessões (%)", fontsize=11)
    ax.set_ylabel("")

    # Mantém a mesma escala em todos os gráficos e reserva
    # espaço adicional para os rótulos.
    ax.set_xlim(0, 120)
    ax.set_xticks(range(0, 101, 20))

    ax.tick_params(axis="both", labelsize=10)

    ax.set_axisbelow(True)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", visible=True, alpha=0.4)

    fig.tight_layout()
    plt.show()

#### Resultados da distribuição das variáveis categóricas

As categorias apresentam frequências desiguais no conjunto de treinamento. Os gráficos mostram o percentual e a quantidade de sessões de cada categoria.

Maio e novembro concentram, juntos, aproximadamente 51,48% das sessões. São observados dez meses distintos; janeiro e abril não aparecem nesse conjunto.

Os visitantes recorrentes representam 85,44% das sessões, enquanto os novos visitantes correspondem a 13,79% e a categoria “Outro” a 0,78%.

A maioria das sessões ocorre fora do fim de semana (77,12%). As sessões de fim de semana representam 22,88%.

Entre as categorias identificadas por códigos, destacam-se:

- Sistema operacional: código 2, com 53,44% das sessões.
- Navegador: código 2, com 64,45%.
- Região: código 1, com 37,92%.
- Tipo de tráfego: código 2, com 31,60%.

Algumas categorias possuem poucos registros, incluindo grupos com apenas uma sessão. Os rótulos permitem identificar essas quantidades mesmo quando as barras são pequenas. Nas próximas análises, suas taxas de compra deverão ser interpretadas considerando o tamanho de cada grupo.

Os códigos numéricos são identificadores de categorias. Nenhum significado específico será atribuído a eles sem o respectivo dicionário de dados.

As frequências descrevem a composição do treinamento e não demonstram, isoladamente, associação com compra ou causalidade. Nenhuma categoria será agrupada ou removida nesta etapa.

---

#### Resultados dos gráficos de frequência

Os gráficos mostram uma distribuição desigual das categorias no conjunto de treinamento.

Em Month, maio e novembro concentram aproximadamente 51,48% das sessões. Essa concentração representa volume de acessos, não necessariamente maior taxa de compra.

Em OperatingSystems e Browser, predomina o código 2, com 53,44% e 64,45% das sessões, respectivamente. Em Region, o código 1 representa 37,92%, enquanto em TrafficType o código 2 corresponde a 31,60%.

Returning_Visitor concentra 85,44% das sessões, indicando predominância de sessões de visitantes recorrentes. As sessões fora do fim de semana representam 77,12%.

As categorias menos frequentes apresentam barras pequenas, algumas quase imperceptíveis na escala de 0 a 100%. Por isso, as tabelas de frequência complementam os gráficos, permitindo consultar suas quantidades e percentuais exatos.

Esses resultados descrevem a composição do treinamento. A associação entre as categorias e a realização de compra será examinada na próxima etapa, considerando também a quantidade de sessões de cada grupo.

---

### 8. Relação entre as variáveis categóricas e a compra

Para cada categoria, calcularemos:

- Quantidade total de sessões.
- Quantidade de sessões com compra.
- Quantidade de sessões sem compra.
- Taxa de compra dentro da categoria.

A taxa de compra corresponde ao número de sessões com compra dividido pelo total de sessões da própria categoria.

Também calcularemos a taxa geral de compra do treinamento como referência para as comparações.

Categorias com poucos registros podem apresentar taxas extremas e instáveis. Os resultados descrevem associações observadas no treinamento e não demonstram causalidade.

In [ ]:
# ============================================================
# TAXA GERAL DE COMPRA NO CONJUNTO DE TREINAMENTO
# ============================================================

train_purchase_rate = df_train[TARGET_COLUMN].mean() * 100

print(
    "Taxa geral de compra no treinamento: "
    + f"{train_purchase_rate:.2f}%".replace(".", ",")
)


# ============================================================
# RELAÇÃO ENTRE CATEGORIAS E REALIZAÇÃO DE COMPRA
# ============================================================

category_purchase_summaries = {}
category_purchase_display_summaries = {}

for column in CATEGORICAL_COLUMNS:
    purchase_summary = (
        df_train
        .groupby(column, observed=True, dropna=False)[TARGET_COLUMN]
        .agg(
            total_sessoes="size",
            sessoes_com_compra="sum",
            taxa_compra_percentual="mean",
        )
        .reset_index()
        .rename(columns={column: "categoria"})
    )

    purchase_summary["sessoes_com_compra"] = (
        purchase_summary["sessoes_com_compra"].astype(int)
    )

    purchase_summary["sessoes_sem_compra"] = (
        purchase_summary["total_sessoes"]
        - purchase_summary["sessoes_com_compra"]
    )

    purchase_summary["taxa_compra_percentual"] *= 100

    purchase_summary = (
        purchase_summary[
            [
                "categoria",
                "total_sessoes",
                "sessoes_com_compra",
                "sessoes_sem_compra",
                "taxa_compra_percentual",
            ]
        ]
        .sort_values(
            by="total_sessoes",
            ascending=False,
            kind="stable",
        )
        .reset_index(drop=True)
    )

    # Preserva os valores originais para as próximas análises.
    category_purchase_summaries[column] = purchase_summary


    # ========================================================
    # TRADUÇÃO DOS VALORES DAS CATEGORIAS
    # ========================================================

    purchase_display = purchase_summary.copy()

    if column in CATEGORY_LABELS:
        purchase_display["categoria"] = (
            purchase_display["categoria"]
            .replace(CATEGORY_LABELS[column])
        )


    # ========================================================
    # APRESENTAÇÃO DA TABELA EM PORTUGUÊS
    # ========================================================

    purchase_display = purchase_display.rename(
        columns={
            "categoria": "Categoria",
            "total_sessoes": "Total de sessões",
            "sessoes_com_compra": "Sessões com compra",
            "sessoes_sem_compra": "Sessões sem compra",
            "taxa_compra_percentual": "Taxa de compra (%)",
        }
    )

    category_purchase_display_summaries[column] = purchase_display

    print(f"\nVariável: {VARIABLE_LABELS.get(column, column)}")

    display(purchase_display)

#### Resultados da relação entre categorias e compra

A taxa geral de compra no conjunto de treinamento é de 15,48%. As diferenças entre categorias devem ser interpretadas considerando o número de sessões de cada grupo.


##### Mês da sessão
Novembro apresenta a maior taxa de compra entre os meses observados: 25,31%, com 523 compras em 2.066 sessões. Outubro também apresenta taxa acima da geral, com 21,78%.

Maio possui o maior volume de sessões, mas sua taxa de compra é de 10,56%, abaixo da taxa geral. Portanto, maior volume de acessos não implica maior proporção de compras.

Fevereiro apresenta a menor taxa observada, de 1,50%, correspondente a duas compras em 133 sessões.

Essas diferenças não comprovam sazonalidade recorrente nem identificam suas causas.


##### Tipo de visitante
As sessões de novos visitantes apresentam taxa de compra de 24,62%, enquanto as de visitantes recorrentes apresentam 14,01%. A diferença é de aproximadamente 10,61 pontos percentuais.

Apesar da menor taxa, os visitantes recorrentes concentram mais sessões com compra em números absolutos: 1.033, devido ao maior volume de sessões.


##### Fim de semana
As sessões de fim de semana apresentam taxa de compra de 17,57%, contra 14,86% nas demais sessões. A diferença é de aproximadamente 2,71 pontos percentuais.


##### Tipo de tráfego
São observadas diferenças entre os códigos de tipo de tráfego. O código 2 apresenta taxa de compra de 22,22%, com 606 compras em 2.727 sessões. Os códigos 1 e 3 apresentam taxas de 10,07% e 8,69%, respectivamente.

O código 8 apresenta taxa de 26,61%, com 62 compras em 233 sessões.


##### Sistema operacional, navegador e região
Em sistema operacional, o código 2 apresenta taxa de compra de 17,61%, enquanto os códigos 1 e 3 apresentam 14,44% e 10,70%, respectivamente.

Em navegador, os códigos mais frequentes, 2 e 1, apresentam taxas de 15,48% e 14,75%, respectivamente.

Em região, as taxas variam de 13,62% a 16,61%, mostrando diferenças descritivas menores entre as categorias.


##### Cuidados de interpretação
Taxas extremas em grupos pequenos exigem cautela. Por exemplo, o código 16 de tipo de tráfego apresenta taxa de 50%, mas possui apenas duas sessões. O código 12 de navegador apresenta 42,86%, com sete sessões.

Os códigos são identificadores de categorias. Seus significados não serão presumidos sem o respectivo dicionário de dados.

As diferenças observadas são descritivas. Não foram realizados testes de significância ou controles por outras variáveis, e os resultados não demonstram causalidade.

Nenhuma variável ou categoria será selecionada, agrupada ou removida apenas com base nessas taxas.

---

### Visualização das taxas de compra por categoria

Os gráficos apresentam a taxa de compra de cada categoria. A linha tracejada indica a taxa geral de compra do treinamento.

Os rótulos incluem o número de sessões de cada grupo (n), permitindo interpretar as taxas considerando o tamanho da amostra.

Todas as categorias serão exibidas, inclusive aquelas com poucos registros. Taxas extremas em grupos pequenos não devem ser interpretadas como evidência de maior ou menor propensão
na população.

As categorias serão apresentadas em ordem de volume de sessões, mantendo a organização das tabelas anteriores.

In [ ]:
# ============================================================
# ORDEM DOS MESES NO CALENDÁRIO
# ============================================================

MONTH_ORDER = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "June": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}


# ============================================================
# CATEGORIAS IDENTIFICADAS POR CÓDIGOS
# ============================================================

CODED_CATEGORY_COLUMNS = [
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
]


# ============================================================
# TAXA GERAL DE COMPRA NO CONJUNTO DE TREINAMENTO
# ============================================================

train_purchase_rate = df_train[TARGET_COLUMN].mean() * 100

general_rate_label = (
    f"{train_purchase_rate:.2f}%".replace(".", ",")
)


# ============================================================
# ORGANIZAÇÃO DAS CATEGORIAS PARA OS GRÁFICOS
# ============================================================

for column in CATEGORICAL_COLUMNS:
    purchase_summary = category_purchase_summaries[column].copy()

    if column == "Month":
        # Organiza os meses antes de traduzir seus nomes.
        purchase_summary = (
            purchase_summary
            .assign(
                ordem_mes=purchase_summary["categoria"].map(
                    MONTH_ORDER
                )
            )
            .sort_values("ordem_mes")
            .drop(columns="ordem_mes")
            .reset_index(drop=True)
        )

    else:
        purchase_summary = (
            purchase_summary
            .sort_values(
                "total_sessoes",
                ascending=False,
                kind="stable",
            )
            .reset_index(drop=True)
        )


    # ========================================================
    # TRADUÇÃO DAS CATEGORIAS PARA APRESENTAÇÃO
    # ========================================================

    category_names = purchase_summary["categoria"].copy()

    if column in CATEGORY_LABELS:
        category_names = category_names.replace(
            CATEGORY_LABELS[column]
        )

    category_names = category_names.astype(str)

    if column in CODED_CATEGORY_COLUMNS:
        category_names = "Código " + category_names


    # ========================================================
    # TAMANHO E ORGANIZAÇÃO DO GRÁFICO
    # ========================================================

    figure_height = max(
        4,
        len(purchase_summary) * 0.45 + 2,
    )

    fig, ax = plt.subplots(
        figsize=(14, figure_height),
    )

    positions = list(range(len(purchase_summary)))

    bars = ax.barh(
        positions,
        purchase_summary["taxa_compra_percentual"],
        color="#2563EB",
        height=0.65,
    )

    ax.set_yticks(
        positions,
        labels=category_names,
    )

    # Exibe a primeira categoria no topo.
    ax.invert_yaxis()


    # ========================================================
    # RÓTULOS COM TAXA, COMPRAS E TOTAL DE SESSÕES
    # ========================================================

    bar_labels = [
        (
            f"{rate:.2f}%".replace(".", ",")
            + " — "
            + f"{purchases:,}".replace(",", ".")
            + " de "
            + f"{total:,}".replace(",", ".")
            + " sessões"
        )
        for rate, purchases, total in zip(
            purchase_summary["taxa_compra_percentual"],
            purchase_summary["sessoes_com_compra"],
            purchase_summary["total_sessoes"],
        )
    ]

    ax.bar_label(
        bars,
        labels=bar_labels,
        padding=6,
        fontsize=10,
    )


    # ========================================================
    # REFERÊNCIA GERAL DO CONJUNTO DE TREINAMENTO
    # ========================================================

    ax.axvline(
        train_purchase_rate,
        color="#B45309",
        linestyle="--",
        linewidth=1.5,
        label=(
            "Taxa geral do treinamento: "
            + general_rate_label
        ),
    )


    # ========================================================
    # TÍTULOS E EIXOS
    # ========================================================

    ax.set_title(
        (
            "Taxa de compra por "
            + VARIABLE_LABELS.get(column, column).lower()
            + " — Treinamento"
        ),
        fontsize=14,
        fontweight="bold",
        pad=16,
    )

    ax.set_xlabel(
        "Taxa de compra (%)",
        fontsize=11,
    )

    ax.set_ylabel("")

    # Reserva espaço à direita para os rótulos.
    # As marcações percentuais permanecem entre 0 e 100%.
    ax.set_xlim(0, 140)
    ax.set_xticks(range(0, 101, 20))

    ax.tick_params(
        axis="both",
        labelsize=10,
    )

    ax.set_axisbelow(True)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", visible=True, alpha=0.4)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        frameon=False,
        fontsize=10,
    )


    # ========================================================
    # NOTA DE INTERPRETAÇÃO E APRESENTAÇÃO
    # ========================================================

    fig.text(
        0.5,
        0.02,
        (
            "Rótulos: taxa de compra — sessões com compra "
            "de total de sessões.\n"
            "Categorias com poucas sessões podem apresentar "
            "taxas instáveis."
        ),
        ha="center",
        fontsize=9,
        color="#475569",
    )

    fig.tight_layout(
        rect=[0, 0.11, 1, 1],
    )

    plt.show()

#### Resultados dos gráficos de taxa de compra

Os gráficos evidenciam diferenças nas taxas de compra em relação à referência geral do treinamento, de 15,48%. Os rótulos mostram
a taxa, a quantidade de sessões com compra e o total de sessões de cada categoria.

Os meses são apresentados na ordem do calendário. Novembro apresenta taxa de 25,31%, enquanto maio, apesar de concentrar o maior volume de sessões, apresenta 10,56%.

As sessões de novos visitantes apresentam taxa de 24,62%, superior aos 14,01% das sessões de visitantes recorrentes.

Nos fins de semana, a taxa é de 17,57%, contra 14,86% nas demais sessões.

O tipo de tráfego apresenta diferenças entre categorias com centenas ou milhares de sessões. O código 2 apresenta taxa de 22,22%, enquanto os códigos 1 e 3 apresentam 10,07% e 8,69%, respectivamente.

As taxas por região permanecem próximas da referência geral, variando de 13,62% a 16,61%.

Os tamanhos dos grupos ajudam a contextualizar taxas extremas. O código 16 de tipo de tráfego apresenta 50%, mas corresponde a apenas uma compra em duas sessões. O código 12 de navegador apresenta 42,86%, com três compras em sete sessões.

As comparações são descritivas. Não foram realizados testes para avaliar significância estatística, e os resultados não demonstram causalidade.

A posição de uma barra em relação à referência geral não determina a importância da variável para o modelo.

---


### 9. Relação entre as variáveis numéricas e a compra

Nesta etapa, comparamos as variáveis numéricas entre sessões sem compra e sessões com compra no conjunto de treinamento.

Para cada variável e classe, calcularemos quantidade de registros, média, mediana, quartis e percentual de valores iguais a zero.

Como as distribuições apresentam assimetria e valores extremos, a interpretação considerará a mediana e os quartis em conjunto com a média.

As diferenças observadas não demonstram causalidade nem determinam, isoladamente, a capacidade preditiva das variáveis.

In [ ]:
# ============================================================
# ESTATÍSTICAS NUMÉRICAS POR RESULTADO DA SESSÃO
# ============================================================

numeric_purchase_summaries = {}

for column in NUMERIC_COLUMNS:
    purchase_summary = (
        df_train
        .groupby(TARGET_COLUMN, observed=True)[column]
        .agg(
            registros="size",
            media="mean",
            mediana="median",
            p25=lambda values: values.quantile(0.25),
            p75=lambda values: values.quantile(0.75),
            percentual_zeros=lambda values: values.eq(0).mean() * 100,
        )
        .reindex([False, True])
        .rename_axis("compra")
        .reset_index()
    )

    purchase_summary["compra"] = purchase_summary["compra"].map(
        {
            False: "Sem compra",
            True: "Com compra",
        }
    )

    # Preserva os nomes internos para as próximas análises.
    numeric_purchase_summaries[column] = purchase_summary


    # ========================================================
    # APRESENTAÇÃO DA TABELA EM PORTUGUÊS
    # ========================================================

    purchase_display = purchase_summary.rename(
        columns={
            "compra": "Resultado da sessão",
            "registros": "Quantidade de sessões",
            "media": "Média",
            "mediana": "Mediana",
            "p25": "P25",
            "p75": "P75",
            "percentual_zeros": "Percentual de zeros (%)",
        }
    )

    print(f"\nVariável: {VARIABLE_LABELS.get(column, column)}")

    purchase_display["Quantidade de sessões"] = (
    purchase_display["Quantidade de sessões"]
    .map(lambda valor: f"{valor:,}".replace(",", "."))
)
    display(purchase_display)

#### Resultados das variáveis numéricas por classe

O conjunto de treinamento contém 7.295 sessões sem compra e 1.336 sessões com compra.


##### Navegação em páginas de produtos
As sessões com compra apresentam mediana de 29 páginas de produtos, contra 16 nas sessões sem compra.

No tempo em páginas de produtos, a mediana é de 1.136,62 nas sessões com compra e de 506,72 nas sessões sem compra. Os quartis também são superiores na classe com compra.

Esses resultados indicam associação entre maior navegação em páginas de produtos e a realização de compra, sem demonstrar uma relação causal.


##### Navegação administrativa e informativa
A quantidade de páginas administrativas apresenta mediana de 2 nas sessões com compra e de 0 nas sessões sem compra. O tempo nessas páginas apresenta medianas de 53,12 e 0, respectivamente.

A quantidade de páginas informativas e o tempo nessas páginas possuem mediana zero nas duas classes, mas apresentam médias maiores e menor proporção de zeros nas sessões com compra.


##### Taxas de rejeição e saída
A taxa de saída apresenta mediana de aproximadamente 0,02 nas sessões com compra e de 0,03 nas sessões sem compra.

A taxa de rejeição apresenta média menor e maior proporção de zeros na classe com compra. Os valores exibidos como 0,00 podem representar números pequenos arredondados pelo formato de apresentação.


##### Valor das páginas
O valor das páginas apresenta diferença expressiva entre as classes: a mediana é de 17,32 nas sessões com compra e de zero nas sessões sem compra.

O percentual de zeros é de 18,41% nas sessões com compra e de 88,75% nas sessões sem compra.

Antes de utilizar essa variável no modelo, será necessário verificar sua construção e disponibilidade no momento da predição, para avaliar possível vazamento de informação.


##### Proximidade de data especial
A proximidade de data especial apresenta mediana zero nas duas classes. O percentual de zeros é de 96,11% nas sessões com compra e de 88,44% nas sessões sem compra.


##### Limitações
As estatísticas descrevem diferenças no treinamento, mas não mostram toda a sobreposição entre as distribuições. Não comprovam causalidade nem determinam isoladamente a importância preditiva das variáveis.

---

#### Comparação das distribuições por classe

Os boxplots apresentam a mediana, os quartis e a dispersão das variáveis em cada classe.

As caixas abrangem do primeiro ao terceiro quartil. Os bigodes seguem o limite de 1,5 vez o intervalo interquartil; pontos além deles não representam necessariamente erros.

As durações e PageValues serão exibidas com log1p para facilitar a leitura. As demais variáveis manterão a escala original. Nenhum registro será removido.

Em variáveis com muitos zeros, a caixa pode ficar reduzida a uma linha. As tabelas anteriores complementam essa leitura.

In [ ]:
# ============================================================
# COMPARAÇÃO DAS VARIÁVEIS NUMÉRICAS POR CLASSE
# ============================================================

fig, axes = plt.subplots(
    nrows=5,
    ncols=2,
    figsize=(16, 22),
)

class_values = [False, True]
class_labels = ["Sem compra", "Com compra"]
class_colors = ["#64748B", "#2563EB"]

for ax, column in zip(axes.flat, NUMERIC_COLUMNS):
    values_by_class = [
        df_train.loc[
            df_train[TARGET_COLUMN].eq(class_value),
            column,
        ].to_numpy()
        for class_value in class_values
    ]


    # ========================================================
    # TRANSFORMAÇÃO APENAS PARA VISUALIZAÇÃO
    # ========================================================

    use_log = column in LOG_VISUALIZATION_COLUMNS

    if use_log:
        values_by_class = [
            np.log1p(values)
            for values in values_by_class
        ]


    # ========================================================
    # CONSTRUÇÃO DOS BOXPLOTS
    # ========================================================

    boxplot = ax.boxplot(
        values_by_class,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.5,
        },
        flierprops={
            "marker": ".",
            "markersize": 3,
            "alpha": 0.25,
        },
    )

    for box, color in zip(boxplot["boxes"], class_colors):
        box.set_facecolor(color)
        box.set_alpha(0.75)


    # ========================================================
    # TÍTULOS E EIXOS EM PORTUGUÊS
    # ========================================================

    ax.set_xticks(
        [1, 2],
        labels=class_labels,
    )

    ax.set_title(
        VARIABLE_LABELS.get(column, column),
        fontsize=13,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Resultado da sessão",
        fontsize=11,
    )

    ax.set_ylabel(
        "Valor transformado: ln(1 + x)"
        if use_log
        else "Valor observado",
        fontsize=11,
    )

    ax.tick_params(axis="both", labelsize=10)

    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=False)
    ax.grid(axis="y", visible=True, alpha=0.4)


# ============================================================
# AJUSTES DA APRESENTAÇÃO
# ============================================================

fig.suptitle(
    "Variáveis Numéricas por Resultado da Sessão — Treinamento",
    fontsize=18,
    fontweight="bold",
)

fig.tight_layout(
    rect=[0, 0, 1, 0.97],
    h_pad=3,
    w_pad=2,
)

plt.show()

#### Resultados dos boxplots por classe

As sessões com compra apresentam medianas superiores na quantidade de páginas administrativas, no tempo nessas páginas, na quantidade de páginas de produtos e no tempo em páginas de produtos.

No tempo em páginas de produtos, a visualização com log1p evidencia o deslocamento da distribuição da classe com compra para valores maiores, embora exista sobreposição entre os grupos.

As taxas de rejeição e saída apresentam valores centrais menores nas sessões com compra. Ainda assim, existem valores compartilhados pelas duas classes.

O valor das páginas apresenta diferença visual expressiva: a caixa das sessões sem compra fica concentrada em zero, enquanto a classe com compra apresenta mediana e quartis superiores. Há, porém, valores positivos nas sessões sem compra e valores zero nas sessões com compra.

A quantidade de páginas informativas, o tempo nessas páginas e a proximidade de data especial apresentam forte concentração em zero. Por isso, algumas caixas ficam reduzidas a uma linha, o que não significa ausência de variação.

Os pontos além dos bigodes representam valores extremos pelo critério do boxplot, não necessariamente erros de registro. Todos os registros foram preservados.

Os gráficos são descritivos e não estabelecem causalidade, significância estatística ou importância preditiva. Permanece necessária a verificação da construção do valor das páginas e de sua disponibilidade no momento da predição.

---

### 10. Análise de correlação

Utilizaremos a correlação de Spearman para examinar associações monotônicas entre as variáveis no conjunto de treinamento.

Spearman utiliza os postos dos valores e é menos sensível à magnitude de valores extremos do que a correlação de Pearson. Valores repetidos, incluindo os zeros, recebem postos médios.

Os coeficientes variam de -1 a 1:

- Valores positivos indicam associação no mesmo sentido.
- Valores negativos indicam associação em sentidos opostos.
- Valores próximos de zero indicam pouca associação monotônica,
  mas não descartam outros tipos de relação.

Revenue será representada por 0 para ausência de compra e 1 para compra, exclusivamente nesta análise.

Os códigos das variáveis categóricas não serão incluídos. As correlações serão calculadas sobre os valores originais, sem aplicação de log1p.

Correlação não demonstra causalidade nem determina, isoladamente, quais variáveis devem integrar o modelo.

In [ ]:
# ============================================================
# PREPARAÇÃO DOS DADOS PARA A CORRELAÇÃO
# ============================================================

correlation_data = df_train[
    NUMERIC_COLUMNS + [TARGET_COLUMN]
].copy()

correlation_data[TARGET_COLUMN] = (
    correlation_data[TARGET_COLUMN].astype(int)
)


# ============================================================
# MATRIZ DE CORRELAÇÃO DE SPEARMAN
# ============================================================

spearman_correlation = correlation_data.corr(method="spearman")

correlation_display = spearman_correlation.rename(
    index=VARIABLE_LABELS,
    columns=VARIABLE_LABELS,
)

fig, ax = plt.subplots(figsize=(16, 13))

sns.heatmap(
    correlation_display,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={"fontsize": 9},
    cbar_kws={"label": "Correlação de Spearman"},
    ax=ax,
)

ax.set_title(
    "Correlação de Spearman — Treinamento",
    pad=16,
)

plt.setp(
    ax.get_xticklabels(),
    rotation=45,
    ha="right",
    rotation_mode="anchor",
)

plt.setp(
    ax.get_yticklabels(),
    rotation=0,
)

fig.tight_layout()
plt.show()


# ============================================================
# ASSOCIAÇÃO DAS VARIÁVEIS NUMÉRICAS COM A COMPRA
# ============================================================

target_correlation = (
    spearman_correlation[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .rename("correlacao_spearman")
    .to_frame()
)

target_correlation["correlacao_absoluta"] = (
    target_correlation["correlacao_spearman"].abs()
)

target_correlation = target_correlation.sort_values(
    "correlacao_absoluta",
    ascending=False,
)

target_correlation.index.name = "variavel"


# ============================================================
# APRESENTAÇÃO DA TABELA 
# ============================================================

target_correlation_display = (
    target_correlation
    .rename(index=VARIABLE_LABELS)
    .rename(
        columns={
            "correlacao_spearman": "Correlação de Spearman",
            "correlacao_absoluta": "Correlação absoluta",
        }
    )
    .rename_axis("Variável")
)

display(target_correlation_display)



### Resultados da análise de correlação

##### Associação com a realização de compra

O valor das páginas apresenta a maior correlação de Spearman com a compra realizada, de aproximadamente 0,64. Esse resultado é coerente com as diferenças observadas nas estatísticas por classe e nos boxplots.

A taxa de saída apresenta a maior associação negativa com a compra, de aproximadamente -0,27. Valores maiores dessa variável tendem a estar associados à ausência de compra.

O tempo em páginas de produtos e a quantidade de páginas de produtos apresentam correlações positivas de aproximadamente 0,22 e 0,20, respectivamente.

A quantidade de páginas administrativas e o tempo nessas páginas apresentam correlações próximas de 0,17. A quantidade de páginas informativas e o tempo nessas páginas apresentam valores próximos de 0,11.

A taxa de rejeição e a proximidade de data especial apresentam correlações negativas de aproximadamente -0,15 e -0,09, respectivamente.

Esses coeficientes descrevem associações individuais. Valores próximos de zero não descartam relações não monotônicas ou interações com outras características.


##### Associação entre as características
São observadas correlações elevadas entre as quantidades de páginas visitadas e seus respectivos tempos de navegação:

- Páginas administrativas e tempo nessas páginas: 0,94.
- Páginas informativas e tempo nessas páginas: 0,95.
- Páginas de produtos e tempo nessas páginas: 0,88.

Essas relações indicam possível compartilhamento de informação, mas não comprovam que as variáveis sejam intercambiáveis.

As taxas de rejeição e saída apresentam correlação positiva de aproximadamente 0,60.

A quantidade de páginas de produtos apresenta correlação negativa de aproximadamente -0,52 com a taxa de saída. O tempo em páginas de produtos apresenta correlação de aproximadamente -0,48 com essa mesma taxa.


##### Implicações para a modelagem
A associação do valor das páginas com a compra reforça a necessidade de verificar sua construção e disponibilidade no momento da predição. A correlação elevada, isoladamente, não comprova vazamento de informação.

As associações elevadas entre características deverão ser consideradas na avaliação dos modelos e na interpretação de seus resultados.

A coluna “Correlação absoluta” representa apenas a magnitude do coeficiente, sem considerar seu sinal. Ela organiza a tabela por intensidade de associação e não representa a importância preditiva das variáveis.


##### Limitações
As correlações foram calculadas exclusivamente no conjunto de treinamento e descrevem associações monotônicas.

Não foram realizados testes de significância estatística. Os resultados não demonstram causalidade nem determinam, isoladamente, quais variáveis devem integrar o modelo.

Nenhuma variável será removida automaticamente com base nesses coeficientes. As decisões serão avaliadas utilizando treinamento e validação, preservando o teste para a avaliação final.

---

### 11. Verificação de registros iguais entre os conjuntos

##### Qualidade dos dados

A base original contém 12.330 sessões e 18 colunas, sem valores ausentes identificados pelo pandas e sem colunas constantes.

Foram encontradas 125 repetições completas além da primeira ocorrência, equivalentes a aproximadamente 1,01% da base. Esses registros foram preservados, pois linhas iguais não comprovam que representam a mesma sessão.

Desconsiderando a variável-alvo, foram identificadas 16 combinações de características compartilhadas entre treinamento e validação, 19 entre treinamento e teste e oito entre validação e teste.

Nas comparações realizadas, os registros envolvidos representam entre 0,35% e 1,24% dos respectivos conjuntos. As contagens não devem ser somadas, pois um mesmo registro pode participar de mais de uma comparação.

Os perfis compartilhados apresentam pouca navegação registrada e diversos valores iguais a zero. A inspeção não permitiu confirmar que representam sessões duplicadas.

Os registros e a divisão atual foram mantidos. A sobreposição será documentada como uma limitação, pois seu impacto no desempenho dos modelos ainda não foi avaliado.


##### Distribuição da variável-alvo
As sessões com compra representam 15,47% da base original e aproximadamente 15,48% do treinamento.

O desbalanceamento exige avaliar o desempenho na classe positiva. A acurácia será acompanhada de precisão, recall, F1, Average Precision e matriz de confusão.

A necessidade de ponderação de classes ou reamostragem será avaliada experimentalmente. Caso utilizada, a reamostragem deverá ocorrer somente nos dados de treinamento de cada ajuste.


##### Características das variáveis numéricas
As variáveis numéricas apresentam assimetria positiva, diferenças de escala e, em vários casos, concentração em zero.

A transformação log1p facilitou a visualização das durações e do valor das páginas. Sua utilização na modelagem dependerá de comparação com outras configurações.

Valores extremos e zeros não foram removidos automaticamente. O pré-processamento deverá considerar o significado das variáveis e as necessidades de cada modelo.


##### Associações com a compra
As sessões com compra apresentam maior quantidade de páginas de produtos visitadas e maior tempo nessas páginas. Também apresentam valores centrais menores nas taxas de rejeição e saída.

Foram observadas diferenças nas taxas de compra por mês, tipo de visitante e tipo de tráfego. Categorias com poucas sessões exigem cautela na interpretação.

Essas associações não demonstram causalidade nem garantem capacidade preditiva em novos dados.


##### Correlação entre características
As contagens de páginas e suas respectivas durações apresentam correlações de Spearman elevadas, entre aproximadamente 0,88 e 0,95.

Essas relações indicam possível compartilhamento de informação. A manutenção ou remoção de características será avaliada na modelagem, sem exclusão automática por correlação.


##### Disponibilidade das informações e possível vazamento
O valor das páginas apresenta a maior correlação observada com a compra, de aproximadamente 0,64.

Antes de sua utilização, será necessário verificar como essa variável foi calculada e se estaria disponível no momento da predição. A correlação elevada, isoladamente, não comprova vazamento de informação.

Também será necessário definir o momento em que a previsão será realizada e verificar se as demais características estarão disponíveis nesse instante.


##### Direcionamentos para a modelagem
- Tratar os códigos de sistema operacional, navegador, região e tipo de tráfego como categorias.
- Ajustar codificações, escalonamentos e transformações exclusivamente nos dados de treinamento de cada ajuste.
- Comparar os modelos com uma referência simples.
- Avaliar as métricas considerando o desbalanceamento.
- Selecionar configurações e limiares com treinamento e validação.
- Reservar o teste para a avaliação final da configuração escolhida.
- Documentar a manutenção dos registros com características idênticas e considerar uma avaliação complementar com separação desses perfis por grupos.
- Verificar a construção e a disponibilidade das características no momento da predição.


##### Limitações da análise
As análises de distribuição e associação foram realizadas no conjunto de treinamento. A inspeção inicial da base original foi utilizada para descrever sua estrutura, qualidade e distribuição da variável-alvo.

Não foram realizados testes de significância ou controles por outras variáveis. As conclusões são descritivas e deverão ser confrontadas com os resultados da modelagem.

---

### 12. Conclusões da análise exploratória e direcionamentos para a modelagem

Nesta etapa, verificamos se existem combinações idênticas de características em diferentes conjuntos de dados.

A comparação considera todas as características, excluindo a variável-alvo. Não serão calculadas métricas de desempenho nem analisadas associações com o alvo na validação ou no teste.

A presença de características iguais não comprova que os registros representam a mesma sessão, mas exige avaliar se a divisão dos dados pode favorecer resultados otimistas.

Nenhum registro será removido e nenhuma divisão será modificada automaticamente.

In [ ]:
# ============================================================
# LOCALIZAÇÃO DOS CONJUNTOS
# ============================================================

DATASET_PATHS = {
    "Treinamento": PROJECT_ROOT / "data" / "processed" / "train.parquet",
    "Validação": PROJECT_ROOT / "data" / "processed" / "validation.parquet",
    "Teste": PROJECT_ROOT / "data" / "processed" / "test.parquet",
}

for dataset_name, dataset_path in DATASET_PATHS.items():
    if not dataset_path.is_file():
        raise FileNotFoundError(
            f"Arquivo de {dataset_name.lower()} não encontrado: "
            f"{dataset_path}"
        )


# ============================================================
# CARREGAMENTO APENAS DAS CARACTERÍSTICAS
# ============================================================

feature_columns = [
    column
    for column in df_train.columns
    if column != TARGET_COLUMN
]

feature_datasets = {
    dataset_name: pd.read_parquet(
        dataset_path,
        columns=feature_columns,
    )
    for dataset_name, dataset_path in DATASET_PATHS.items()
}


# ============================================================
# COMPARAÇÃO ENTRE OS CONJUNTOS
# ============================================================

dataset_pairs = [
    ("Treinamento", "Validação"),
    ("Treinamento", "Teste"),
    ("Validação", "Teste"),
]

overlap_results = []

for first_name, second_name in dataset_pairs:
    first_features = feature_datasets[first_name].drop_duplicates()
    second_features = feature_datasets[second_name].drop_duplicates()

    shared_features = first_features.merge(
        second_features,
        on=feature_columns,
        how="inner",
        validate="one_to_one",
    )

    overlap_results.append(
        {
            "Primeiro conjunto": first_name,
            "Segundo conjunto": second_name,
            "Combinações de características em comum": len(
                shared_features
            ),
        }
    )

overlap_summary = pd.DataFrame(overlap_results)

display(overlap_summary)

#### Resultados da verificação entre os conjuntos

Foram identificadas combinações idênticas de características
entre os conjuntos:

- Treinamento e validação: 16 combinações.
- Treinamento e teste: 19 combinações.
- Validação e teste: 8 combinações.

A comparação desconsiderou a variável-alvo e contabilizou combinações distintas de características, não o total de registros envolvidos.

Essas coincidências não comprovam que os registros representam a mesma sessão. Entretanto, indicam uma possível fonte de avaliação excessivamente otimista, caso correspondam a sessões duplicadas distribuídas entre os conjuntos.

Nenhum registro foi removido nesta etapa. Antes da modelagem, será necessário investigar os registros envolvidos e documentar a decisão sobre sua manutenção ou sobre uma eventual revisão da divisão dos dados.

---

In [ ]:
# ============================================================
# REGISTROS COM CARACTERÍSTICAS PRESENTES EM OUTROS CONJUNTOS
# ============================================================

overlap_detail_results = []

for first_name, second_name in dataset_pairs:
    first_dataset = feature_datasets[first_name]
    second_dataset = feature_datasets[second_name]

    shared_features = (
        first_dataset
        .drop_duplicates()
        .merge(
            second_dataset.drop_duplicates(),
            on=feature_columns,
            how="inner",
            validate="one_to_one",
        )
    )

    first_matches = first_dataset.merge(
        shared_features,
        on=feature_columns,
        how="inner",
        validate="many_to_one",
    )

    second_matches = second_dataset.merge(
        shared_features,
        on=feature_columns,
        how="inner",
        validate="many_to_one",
    )

    overlap_detail_results.append(
        {
            "Primeiro conjunto": first_name,
            "Segundo conjunto": second_name,
            "Combinações em comum": len(shared_features),
            "Registros envolvidos no primeiro": len(first_matches),
            "Percentual do primeiro (%)": (
                len(first_matches) / len(first_dataset) * 100
            ),
            "Registros envolvidos no segundo": len(second_matches),
            "Percentual do segundo (%)": (
                len(second_matches) / len(second_dataset) * 100
            ),
        }
    )


# ============================================================
# APRESENTAÇÃO DO DETALHAMENTO
# ============================================================

overlap_detail_summary = pd.DataFrame(overlap_detail_results)

display(overlap_detail_summary.round(2))

#### Detalhamento dos registros envolvidos

As 16 combinações de características compartilhadas entre treinamento e validação envolvem 30 registros do treinamento (0,35%) e 23 da validação (1,24%).

Entre treinamento e teste, as 19 combinações compartilhadas envolvem 41 registros do treinamento (0,48%) e 22 do teste (1,19%).

Entre validação e teste, as oito combinações compartilhadas envolvem 12 registros da validação (0,65%) e 11 do teste (0,59%).

As contagens não devem ser somadas, pois um mesmo registro pode participar de mais de uma comparação.

Os percentuais indicam sobreposição limitada nas comparações realizadas. Entretanto, sua magnitude não determina o impacto sobre a avaliação dos modelos.

Características idênticas não comprovam que os registros correspondem à mesma sessão. A inspeção dos perfis compartilhados é apresentada a seguir.

---

In [ ]:
# ============================================================
# ATUALIZAÇÃO DOS RÓTULOS
# ============================================================

VARIABLE_LABELS.update(
    {
        "Month": "Mês",
        "OperatingSystems": "Sistema operacional",
        "Browser": "Navegador",
        "Region": "Região",
        "TrafficType": "Tipo de tráfego",
        "VisitorType": "Tipo de visitante",
        "Weekend": "Fim de semana",
    }
)


# ============================================================
# IDENTIFICAÇÃO DAS COMBINAÇÕES COMPARTILHADAS
# ============================================================

shared_profile_tables = {}

for first_name, second_name in dataset_pairs:
    first_dataset = feature_datasets[first_name]
    second_dataset = feature_datasets[second_name]

    first_counts = (
        first_dataset
        .groupby(
            feature_columns,
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="registros_primeiro")
    )

    second_counts = (
        second_dataset
        .groupby(
            feature_columns,
            observed=True,
            dropna=False,
        )
        .size()
        .reset_index(name="registros_segundo")
    )

    shared_profiles = first_counts.merge(
        second_counts,
        on=feature_columns,
        how="inner",
        validate="one_to_one",
    )

    shared_profiles["total_registros"] = (
        shared_profiles["registros_primeiro"]
        + shared_profiles["registros_segundo"]
    )

    shared_profiles = (
        shared_profiles
        .sort_values(
            "total_registros",
            ascending=False,
            kind="stable",
        )
        .reset_index(drop=True)
    )

    shared_profile_tables[(first_name, second_name)] = (
        shared_profiles
    )


# ============================================================
# APRESENTAÇÃO DOS PERFIS EM PORTUGUÊS
# ============================================================

for dataset_pair, shared_profiles in shared_profile_tables.items():
    first_name, second_name = dataset_pair

    display_profiles = shared_profiles.copy()

    for column, translations in CATEGORY_LABELS.items():
        if column in display_profiles.columns:
            display_profiles[column] = (
                display_profiles[column]
                .astype(object)
                .replace(translations)
            )

    display_profiles = display_profiles.rename(
        columns={
            **VARIABLE_LABELS,
            "registros_primeiro": f"Registros — {first_name}",
            "registros_segundo": f"Registros — {second_name}",
            "total_registros": "Total de registros no par",
        }
    )

    count_columns = [
        f"Registros — {first_name}",
        f"Registros — {second_name}",
        "Total de registros no par",
    ]

    feature_display_columns = [
        VARIABLE_LABELS.get(column, column)
        for column in feature_columns
    ]

    display_profiles = display_profiles[
        count_columns + feature_display_columns
    ]

    print(
        f"\n{first_name} × {second_name}: "
        f"{len(shared_profiles)} combinações compartilhadas"
    )

    with pd.option_context(
        "display.max_columns", None,
        "display.max_rows", None,
    ):
        display(display_profiles)

#### Características dos perfis compartilhados

Os perfis compartilhados apresentam ausência de navegação em páginas administrativas e informativas, além de uma ou duas páginas de produtos registradas.

As durações são exibidas como zero na precisão apresentada. As taxas de rejeição e saída são de 0,20, e o valor das páginas é zero.

As combinações diferem principalmente nas características categóricas, como mês, sistema operacional, navegador, região e tipo de tráfego.

Esses padrões são compatíveis com sessões de pouca navegação registrada. Diferentes sessões podem apresentar as mesmas características, especialmente quando há muitos valores iguais a zero.

A comparação foi realizada com os valores armazenados, antes do arredondamento utilizado na apresentação.

A inspeção não permite confirmar que as coincidências representam a mesma sessão. Por esse motivo, os registros e a divisão atual serão mantidos nesta etapa.

A sobreposição será documentada como uma limitação. Uma avaliação complementar com separação por grupos de características idênticas poderá ser realizada na modelagem, caso seja necessário avaliar a sensibilidade dos resultados.

---